# 02. 특보 이벤트 수 세기 (event counts)

**이 노트북이 하는 일**

특보 종류(폭염·한파·호우·대설)별로 **연속해서 켜진 날을 하나의 이벤트**로 묶고, 각 이벤트가 다른 사건과 겹치는지 표시한 뒤, 겹침을 단계별로 제외해 **분석에 쓸 수 있는 "깨끗한 이벤트" 수**를 셉니다 (피드백 1: 날씨 종류별로 사용 가능한 사건 수 표, 적은 칸은 "참고용" 표시).

- **이벤트**: 같은 종류의 특보가 연속으로 켜진 날들 (하루라도 꺼지면 끊김). 지속일수 = 종료일 − 시작일 + 1.
- **기간**: 시작일 기준으로 train(2014~2023) / validation(2024) / test(2025). 기간 경계에 걸친 이벤트도 시작일 기준입니다.
- **겹침 확인 창**: 이벤트 시작일부터 **종료일 + `ALERT_MASK_AFTER_DAYS`(7)일**까지. 이 창 안에서 아래 사건이 있으면 "겹침"입니다.
  1. 파일 경계: 시작일 또는 종료일을 알 수 없음 (특보 파일은 2014-01-08부터 시작하고 2025-12-31에서 끝남)
  2. 코로나(2020-02-01~2022-04-30), 메르스(2015-05-20~07-28)
  3. 공휴일, 명절 ±2일, 근로자의 날(5/1), 파업
  4. 다른 종류의 특보
  5. 계단 더미 시행일(요금 인상, 기후동행카드, 신림선, 신분당선) ±30일

> 특보는 서울 전체 기준이라 9개 역이 같은 이벤트를 공유합니다. 역마다 달라지는 것은 시청역의 집회일과 서울대입구·신사의 개통 더미뿐이라, 이벤트 수는 역 유형과 상관없이 같게 셉니다 (시청 집회 겹침은 참고용 열로만 표시). **이용량(`total`)은 쓰지 않고**, 2025(test) 이벤트도 세지만 **모델링에는 07번 전까지 쓰지 않습니다.**

**입력 파일**

- `data/processed/model_input_9stations.csv` (특보 열, 가리기 열, 계단 더미 열)

**출력 파일**

- `results/events.csv` (이벤트별 표: 종류, 시작일, 종료일, 지속일수, 기간, 겹침 표시)
- `results/event_counts.csv` (종류 × 단계 × 기간별 이벤트 수)
- `results/event_overlap_reasons.csv` (겹침 사유별 이벤트 수)


**설정 불러오기**: Colab이면 저장소를 clone해 경로를 맞추고, `00_settings.ipynb`의 설정을 불러옵니다 (로컬이면 clone은 건너뜀).

In [1]:
# Colab에서 실행하면 저장소를 clone하고 notebooks/로 이동합니다 (로컬이면 건너뜁니다).
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/TimeSeries'):
        subprocess.run(['git', 'clone', 'https://github.com/Eui-seung/TimeSeries.git', '/content/TimeSeries'], check=True)
    os.chdir('/content/TimeSeries/notebooks')

# 설정(경로, 역, 열 이름, 기간, 차수 등)을 불러옵니다.
%run ./00_settings.ipynb

필요한 패키지를 불러옵니다.

In [2]:
import numpy as np
import pandas as pd

**날짜별 달력 표 만들기**: 모델 입력 표에서 날짜마다 특보, 가리기 열(공휴일·명절·근로자의 날·코로나·메르스·파업), 계단 더미를 가져옵니다. 이 열들은 서울 전체 기준이라 9개 역이 같은 값이어야 하므로 같은지 확인하고, 시청 행을 대표로 씁니다.

In [3]:
mi = pd.read_csv(PROC / 'model_input_9stations.csv', parse_dates=['date'])
same_cols = [c for c in MASK_COLS if c != 'rally_cityhall'] + list(EVENT_TYPES.values()) + ['any_alert']
nun = mi.groupby('date')[same_cols].nunique().max()            # 날짜마다 9개 역의 값이 몇 종류인가
assert (nun <= 1).all(), nun[nun > 1]
cal = mi[mi.station == STATIONS[0]].set_index('date').sort_index()[same_cols + ['rally_cityhall']]
print('날짜', cal.index.min().date(), '~', cal.index.max().date(), f'({len(cal)}일) | 특보 여부를 모르는 날(alert_unknown):', int(cal.alert_unknown.sum()), '일')

날짜 2014-01-01 ~ 2025-12-31 (4383일) | 특보 여부를 모르는 날(alert_unknown): 7 일


**계단 더미 시행일 ±30일 구간**: 더미 열이 처음 1이 되는 날을 시행일로 보고, 그 앞뒤 `EVENT_STEP_PM_DAYS`(30)일을 "겹침 구간"으로 표시합니다 (역별 더미는 그 역의 행에서 찾음).

In [4]:
step_dates = {}
for c in STEP_COLS:
    step_dates[c] = mi.loc[mi[c] == 1, 'date'].min()
for st, c in STATION_STEP.items():
    step_dates[c] = mi.loc[(mi.station == st) & (mi[c] == 1), 'date'].min()
near_step = pd.Series(False, index=cal.index)
for d in step_dates.values():
    near_step |= (cal.index >= d - pd.Timedelta(days=EVENT_STEP_PM_DAYS)) & (cal.index <= d + pd.Timedelta(days=EVENT_STEP_PM_DAYS))
pd.Series({k: str(v.date()) for k, v in step_dates.items()}, name='시행일')

fare_2015         2015-06-27
fare_2023         2023-10-07
fare_2025         2025-06-28
climate_card      2024-01-27
sillim_line       2022-05-28
sinbundang_ext    2022-05-28
Name: 시행일, dtype: object

**`make_runs`**: 한 종류의 특보 열에서 연속해서 켜진 구간(이벤트)을 찾습니다. 특보 여부를 모르는 날(NaN)은 이벤트를 끊고, 구간의 앞날 또는 뒷날이 모르는 날이거나 표의 끝이면 시작·종료일을 알 수 없는 이벤트로 표시합니다.

In [5]:
def make_runs(s):
    """s: 날짜 인덱스의 특보 열(1=켜짐, 0=꺼짐, NaN=모름). 반환: [(시작일, 종료일, 시작불명, 종료불명), ...]"""
    active = (s == 1)
    gid = (active != active.shift(fill_value=False)).cumsum()
    out = []
    for _, g in active[active].groupby(gid[active]):
        a, b = g.index.min(), g.index.max()
        before, after = a - pd.Timedelta(days=1), b + pd.Timedelta(days=1)
        left = (a == s.index.min()) or (before in s.index and pd.isna(s.loc[before]))
        right = (b == s.index.max()) or (after in s.index and pd.isna(s.loc[after]))
        out.append((a, b, bool(left), bool(right)))
    return out

**`period_of`**: 이벤트의 시작일로 기간을 정합니다 (2023-12-31까지 train, 2024년 validation, 2025년 test).

In [6]:
def period_of(d):
    if d <= pd.Timestamp(TRAIN_END):
        return 'train'
    return 'validation' if d.year == VALID else 'test'

**이벤트 표 만들기**: 종류마다 이벤트를 찾고, 각 이벤트의 확인 창(시작일 ~ 종료일+7일) 안에서 겹침 사유를 확인합니다. 겹침은 사유별 True/False 열로 남기고, 아래 순서대로 처음 걸리는 사유를 `제외단계`로 기록합니다.

In [7]:
REASONS = ['파일경계', '코로나·메르스', '공휴일·명절·근로자의날·파업', '다른특보', '계단더미±30일']     # 제외 순서
rows = []
for kind, col in EVENT_TYPES.items():
    others = [c for k, c in EVENT_TYPES.items() if k != kind]
    for n, (a, b, left, right) in enumerate(make_runs(cal[col]), start=1):
        w = slice(a, min(b + pd.Timedelta(days=ALERT_MASK_AFTER_DAYS), cal.index.max()))       # 겹침 확인 창
        win = cal.loc[w]
        f = {
            '파일경계': bool(left or right),
            '코로나·메르스': bool(win.covid.any() or win.mers.any()),
            '공휴일·명절·근로자의날·파업': bool(win.holiday.any() or win.myeongjeol_pm2.any() or win.labor_day.any() or win.strike.any()),
            '다른특보': bool((win[others] == 1).any().any()),
            '계단더미±30일': bool(near_step.loc[w].any()),
        }
        rows.append({'이벤트번호': f'{kind}-{n:02d}', '종류': kind, '시작일': a.date(), '종료일': b.date(), '지속일수': (b - a).days + 1, '기간': period_of(a),
                     '시작불명': left, '종료불명': right, '코로나': bool(win.covid.any()), '메르스': bool(win.mers.any()), '공휴일': bool(win.holiday.any()),
                     '명절±2': bool(win.myeongjeol_pm2.any()), '근로자의날': bool(win.labor_day.any()), '파업': bool(win.strike.any()), '다른특보': f['다른특보'],
                     '계단더미±30일': f['계단더미±30일'], '시청집회(참고)': bool(win.rally_cityhall.any()),
                     '기간경계걸침': period_of(b) != period_of(a),
                     '제외단계': next((i + 1 for i, r in enumerate(REASONS) if f[r]), 0), '깨끗함': not any(f.values())})
ev = pd.DataFrame(rows)
ev['제외사유'] = ev.apply(lambda r: ', '.join(x for x in REASONS if x == '파일경계' and (r.시작불명 or r.종료불명)
                                          or x == '코로나·메르스' and (r.코로나 or r.메르스)
                                          or x == '공휴일·명절·근로자의날·파업' and (r.공휴일 or r['명절±2'] or r.근로자의날 or r.파업)
                                          or x == '다른특보' and r.다른특보 or x == '계단더미±30일' and r['계단더미±30일']), axis=1)
print('이벤트', len(ev), '개 | 깨끗한 이벤트', int(ev.깨끗함.sum()), '개')
ev.head(8)

이벤트 248 개 | 깨끗한 이벤트 60 개


,이벤트번호,종류,시작일,종료일,지속일수,기간,시작불명,종료불명,코로나,메르스,...,명절±2,근로자의날,파업,다른특보,계단더미±30일,시청집회(참고),기간경계걸침,제외단계,깨끗함,제외사유
0,폭염-01,폭염,2014-07-09,2014-07-11,3,train,False,False,False,False,...,False,False,False,True,False,False,False,4,False,다른특보
1,폭염-02,폭염,2014-07-30,2014-08-02,4,train,False,False,False,False,...,False,False,False,False,False,False,False,0,True,
2,폭염-03,폭염,2015-07-10,2015-07-11,2,train,False,False,False,True,...,False,False,False,True,True,False,False,2,False,"코로나·메르스, 다른특보, 계단더미±30일"
3,폭염-04,폭염,2015-08-06,2015-08-10,5,train,False,False,False,False,...,False,False,False,True,False,False,False,3,False,"공휴일·명절·근로자의날·파업, 다른특보"
4,폭염-05,폭염,2015-08-17,2015-08-18,2,train,False,False,False,False,...,False,False,False,False,False,False,False,0,True,
5,폭염-06,폭염,2016-05-20,2016-05-22,3,train,False,False,False,False,...,False,False,False,False,False,False,False,0,True,
6,폭염-07,폭염,2016-07-08,2016-07-11,4,train,False,False,False,False,...,False,False,False,False,False,False,False,0,True,
7,폭염-08,폭염,2016-07-19,2016-07-26,8,train,False,False,False,False,...,False,False,False,True,False,False,False,4,False,다른특보


**이벤트 표 점검**: (1) 종류별 이벤트의 지속일수 합이 특보가 켜진 날 수와 같은지, (2) 같은 종류 이벤트끼리 날짜가 겹치지 않는지 확인합니다.

In [8]:
active_days = {k: int((cal[c] == 1).sum()) for k, c in EVENT_TYPES.items()}
got = ev.groupby('종류').지속일수.sum().to_dict()
print('특보 켜진 날 수:', active_days)
print('이벤트 지속일수 합:', got)
assert all(active_days[k] == got[k] for k in EVENT_TYPES)
for k, g in ev.groupby('종류'):
    g = g.sort_values('시작일')
    assert (pd.to_datetime(g.시작일.iloc[1:].values) > pd.to_datetime(g.종료일.iloc[:-1].values)).all(), k
print('점검 통과: 지속일수 합이 맞고 같은 종류 이벤트끼리 겹치지 않음')

특보 켜진 날 수: {'폭염': 386, '한파': 172, '호우': 150, '대설': 48}
이벤트 지속일수 합: {'대설': 48, '폭염': 386, '한파': 172, '호우': 150}
점검 통과: 지속일수 합이 맞고 같은 종류 이벤트끼리 겹치지 않음


**이벤트 표 저장**: `results/events.csv`

In [9]:
RESULTS.mkdir(exist_ok=True)
ev.to_csv(RESULTS / 'events.csv', index=False, encoding='utf-8-sig')
print('events.csv:', ev.shape)

events.csv: (248, 21)


**단계별 이벤트 수 표**: 종류 × 기간별로 전체 이벤트에서 시작해, 겹침 사유를 한 단계씩 누적해서 제외한 뒤 남는 이벤트 수를 셉니다. 마지막 행이 "깨끗한 이벤트"입니다. `train+val`은 07번 전에 모델링에 쓸 수 있는 train과 validation의 합입니다.

In [10]:
STEP_LABELS = ['0 전체 이벤트', '1 파일 경계(시작·종료 불명) 제외', '2 + 코로나·메르스 겹침 제외', '3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외',
               '4 + 다른 종류 특보 겹침 제외', '5 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)']
PERIODS = ['train', 'validation', 'test']
rows = []
for kind in EVENT_TYPES:
    g = ev[ev.종류 == kind]
    for k, lab in enumerate(STEP_LABELS):
        keep = g[(g.제외단계 == 0) | (g.제외단계 > k)]          # k단계까지 제외하고 남는 이벤트
        cnt = keep.기간.value_counts().reindex(PERIODS, fill_value=0)
        rows.append({'종류': kind, '단계': lab, **cnt.to_dict(), 'train+val': int(cnt['train'] + cnt['validation']), '합계': int(cnt.sum())})
counts = pd.DataFrame(rows)
counts.to_csv(RESULTS / 'event_counts.csv', index=False, encoding='utf-8-sig')
counts.shape

(24, 7)

**표 보기 (10개 미만은 ⚠ 표시)**: 각 종류의 단계별 이벤트 수입니다. 마지막 행(깨끗한 이벤트)의 칸에서 `MIN_EVENTS`(10)개 미만이면 ⚠를 붙입니다.

In [11]:
def mark(v, last):
    return f'{v} ⚠' if (last and v < MIN_EVENTS) else str(v)
show = counts.copy()
last = show.단계.str.startswith('5')
for c in PERIODS + ['train+val', '합계']:
    show[c] = [mark(v, l) for v, l in zip(counts[c], last)]
show.set_index(['종류', '단계'])

train validation test train+val   합계
종류 단계                                                                       
폭염 0 전체 이벤트                                46          5    5        51   56
   1 파일 경계(시작·종료 불명) 제외                    46          5    5        51   56
   2 + 코로나·메르스 겹침 제외                       39          5    5        44   49
   3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외            30          2    4        32   36
   4 + 다른 종류 특보 겹침 제외                       9          1    0        10   10
   5 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)   9 ⚠        1 ⚠  0 ⚠        10   10
한파 0 전체 이벤트                                49          1    6        50   56
   1 파일 경계(시작·종료 불명) 제외                    48          1    5        49   54
   2 + 코로나·메르스 겹침 제외                       34          1    5        35   40
   3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외            19          1    4        20   24
   4 + 다른 종류 특보 겹침 제외                      17          1    2        18   20
   5 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)    17        0 ⚠  2 ⚠        17   19
호우 0 전체 이벤트                                80         10   10        90  100
   1 파일 경계(시작·종료 불명) 제외                    80         10   10        90  100
   2 + 코로나·메르스 겹침 제외                       57         10   10        67   77
   3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외            45          7    9        52   61
   4 + 다른 종류 특보 겹침 제외                      16          2    4        18   22
   5 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)    15        2 ⚠  3 ⚠        17   20
대설 0 전체 이벤트                                25          5    6        30   36
   1 파일 경계(시작·종료 불명) 제외                    25          5    6        30   36
   2 + 코로나·메르스 겹침 제외                       15          5    6        20   26
   3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외             9          4    5        13   18
   4 + 다른 종류 특보 겹침 제외                       7          4    3        11   14
   5 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)   7 ⚠        1 ⚠  3 ⚠       8 ⚠   11

**종류별 판정**: 깨끗한 이벤트 수(train+validation 기준)가 `MIN_EVENTS`(10)개 이상이면 "충분", 미만이면 "참고용"으로 표시합니다. test(2025)는 07번 전까지 모델링에 쓰지 않으므로 판정에서 뺍니다.

In [12]:
fin = counts[counts.단계.str.startswith('5')].set_index('종류')
judge = pd.DataFrame({'전체 이벤트': counts[counts.단계.str.startswith('0')].set_index('종류')['합계'],
                      '깨끗한 이벤트(train+val)': fin['train+val'], 'train': fin['train'], 'validation': fin['validation'], 'test(참고)': fin['test']})
judge['판정'] = np.where(judge['깨끗한 이벤트(train+val)'] >= MIN_EVENTS, '충분', '참고용(10개 미만)')
judge

,전체 이벤트,깨끗한 이벤트(train+val),train,validation,test(참고),판정
종류,,,,,,
폭염,56,10,9,1,0,충분
한파,56,17,17,0,2,충분
호우,100,17,15,2,3,충분
대설,36,8,7,1,3,참고용(10개 미만)


**겹침 사유별 이벤트 수**: 사유마다 그 사유에 걸린 이벤트가 몇 개인지 셉니다 (한 이벤트가 여러 사유에 걸리면 각각 셈. 위의 누적 제외와 달리 사유끼리 겹쳐서 세는 표). `results/event_overlap_reasons.csv`로 저장합니다.

In [13]:
flag_cols = ['시작불명', '종료불명', '코로나', '메르스', '공휴일', '명절±2', '근로자의날', '파업', '다른특보', '계단더미±30일', '시청집회(참고)']
reasons = ev.groupby('종류')[flag_cols].sum().astype(int)
reasons.insert(0, '전체', ev.groupby('종류').size())
reasons = reasons.reindex(list(EVENT_TYPES))
reasons.to_csv(RESULTS / 'event_overlap_reasons.csv', encoding='utf-8-sig')
reasons

,전체,시작불명,종료불명,코로나,메르스,공휴일,명절±2,근로자의날,파업,다른특보,계단더미±30일,시청집회(참고)
종류,,,,,,,,,,,,
폭염,56,0,0,6,1,16,2,0,0,42,5,0
한파,56,1,1,14,0,18,6,0,2,17,3,5
호우,100,0,0,21,2,18,6,0,1,55,9,0
대설,36,0,0,10,0,11,5,0,0,13,6,6


**연도별 이벤트 수**: 종류별로 전체 이벤트와 깨끗한 이벤트가 연도마다 몇 개인지 봅니다 (시작일 기준).

In [14]:
ev['연도'] = pd.to_datetime(ev.시작일).dt.year
yr = ev.pivot_table(index='연도', columns='종류', values='이벤트번호', aggfunc='count', fill_value=0)[list(EVENT_TYPES)]
yr_clean = ev[ev.깨끗함].pivot_table(index='연도', columns='종류', values='이벤트번호', aggfunc='count', fill_value=0).reindex(index=yr.index, columns=yr.columns, fill_value=0)
pd.concat({'전체': yr, '깨끗한 이벤트': yr_clean}, axis=1)

전체            깨끗한 이벤트         
종류   폭염  한파  호우 대설      폭염 한파 호우 대설
연도                                 
2014  2   3   6  2       1  1  3  2
2015  3   1   3  2       1  1  0  2
2016  4   4   4  1       2  4  0  0
2017  7   2  11  2       0  0  3  0
2018  3   7   5  1       0  4  3  1
2019  8   6   6  1       3  4  1  1
2020  5   3  12  2       0  0  0  0
2021  1  10   9  6       0  0  0  0
2022  7   5  12  4       1  0  2  0
2023  6   8  12  4       1  3  3  1
2024  5   1  10  5       1  0  2  1
2025  5   6  10  6       0  2  3  3

**깨끗한 이벤트 목록**: 분석에 쓸 수 있는 이벤트를 종류별로 모두 보여 줍니다 (04번 노트북의 입력).

In [15]:
ev[ev.깨끗함][['이벤트번호', '종류', '시작일', '종료일', '지속일수', '기간', '시청집회(참고)']].reset_index(drop=True)

,이벤트번호,종류,시작일,종료일,지속일수,기간,시청집회(참고)
0,폭염-02,폭염,2014-07-30,2014-08-02,4,train,False
1,폭염-05,폭염,2015-08-17,2015-08-18,2,train,False
2,폭염-06,폭염,2016-05-20,2016-05-22,3,train,False
3,폭염-07,폭염,2016-07-08,2016-07-11,4,train,False
4,폭염-20,폭염,2019-05-24,2019-05-25,2,train,False
5,폭염-21,폭염,2019-06-25,2019-06-26,2,train,False
6,폭염-27,폭염,2019-08-19,2019-08-20,2,train,False
7,폭염-40,폭염,2022-08-20,2022-08-21,2,train,False
8,폭염-41,폭염,2023-06-18,2023-06-19,2,train,False
9,폭염-47,폭염,2024-06-19,2024-06-21,3,validation,False


**참고: 기간 경계에 걸친 이벤트와 가장 긴 이벤트**: 시작일 기준으로 기간을 정했기 때문에 train과 validation 경계(2023-12-31)에 걸친 이벤트가 있는지 확인합니다.

In [16]:
print('기간 경계에 걸친 이벤트:', ev[ev.기간경계걸침][['이벤트번호', '시작일', '종료일', '기간']].to_dict('records'))
ev.sort_values('지속일수', ascending=False).head(5)[['이벤트번호', '시작일', '종료일', '지속일수', '기간', '깨끗함']]

기간 경계에 걸친 이벤트: []


,이벤트번호,시작일,종료일,지속일수,기간,깨끗함
48,폭염-49,2024-07-24,2024-08-31,39,validation,False
17,폭염-18,2018-07-12,2018-08-18,38,train,False
32,폭염-33,2021-07-10,2021-08-13,35,train,False
8,폭염-09,2016-07-31,2016-08-25,26,train,False
52,폭염-53,2025-07-21,2025-08-06,17,test,False
